# 🧪 Supplement Detection V3 Training

Version3 데이터셋을 사용한 YOLO11m 학습 노트북입니다.

## 데이터셋 구성
- 기존 Roboflow 데이터
- huseong 크롤링 데이터 (검토 완료)
- COCO 부정 샘플 (bottle 없는 이미지)
- SAMPLE 배경 이미지

## 1. 환경 설정

In [ ]:
# Ultralytics 설치
!pip install ultralytics -q

# GPU 확인
!nvidia-smi

In [ ]:
import os
import shutil
from pathlib import Path
from ultralytics import YOLO

# 데이터셋 경로 확인 (압축 해제 후 현재 폴더 기준)
data_path = Path('data.yaml')
print(f"데이터셋 경로: {data_path}")
print(f"존재 여부: {data_path.exists()}")

# 데이터셋 정보 출력
if data_path.exists():
    with open(data_path, 'r') as f:
        print(f.read())

## 2. 데이터셋 통계 확인

In [ ]:
# 각 폴더별 이미지 수 확인
for split in ['train', 'valid', 'test']:
    img_dir = Path(f'{split}/images')
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        print(f"{split}: {count}개 이미지")

## 3. 모델 학습

In [ ]:
# YOLO11m 모델 로드
model = YOLO('yolo11m.pt')

# 학습 실행
results = model.train(
    data='data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    patience=30,
    workers=8,
    save=True,
    project='Supplement_Detection',
    name='Version3',
    exist_ok=True,
    pretrained=True,
    # 데이터 증강
    mosaic=1.0,
    mixup=0.1,
)

## 4. 학습 결과 확인

In [ ]:
# 결과 그래프 표시
from IPython.display import Image, display

results_img = Path('Supplement_Detection/Version3/results.png')
if results_img.exists():
    display(Image(filename=str(results_img)))

## 5. 최종 모델 저장 및 압축

In [ ]:
# best.pt를 Version3.pt로 복사
best_path = Path('Supplement_Detection/Version3/weights/best.pt')
final_path = Path('Supplement_Detection/Version3/weights/Version3.pt')

if best_path.exists():
    shutil.copy(best_path, final_path)
    print(f"🎉 최종 모델: {final_path}")
else:
    print("⚠️ best.pt를 찾을 수 없습니다.")

In [ ]:
# 결과 폴더 전체 압축
result_folder = 'Supplement_Detection/Version3'
output_filename = 'Supplement_Version3_Final_Results'

if os.path.exists(result_folder):
    shutil.make_archive(output_filename, 'zip', result_folder)
    print(f"✅ 압축 완료: {output_filename}.zip")
    print("💡 왼쪽 파일 탐색기에서 해당 zip 파일을 다운로드하세요!")
else:
    print(f"❌ '{result_folder}' 폴더를 찾을 수 없습니다.")

## 6. 테스트 추론 (선택)

In [ ]:
# 학습된 모델로 테스트 이미지 추론
test_model = YOLO('Supplement_Detection/Version3/weights/best.pt')

# 테스트 이미지 하나로 추론
test_images = list(Path('test/images').glob('*.jpg'))[:5]
if test_images:
    results = test_model(test_images, conf=0.25)
    for r in results:
        r.show()